# Tutorial 1, Your turn again: bike rentals  🚲

> **A bike-share operator wants to plan how many bikes to have ready each day.**
> **Does the day's temperature predict how many get rented?**

Same job as Armand's Pizza, second dataset, real this time (not a made-up teaching example): 731 real days of a real bike-share scheme. Same mechanics, so this should feel familiar, and the same habit still matters most: **predict before you compute.**

**How to work**
- First, **File → Save a copy in Drive**, so your work is kept.
- Solo or in pairs. Let **Claude write and explain the code**; *you* own the hypothesis and the interpretation.
- Interrogate every answer Claude gives you. A model you can't question is a model you can't trust.
- The example prompts (in quotes) are starting points, make them your own.

In [ ]:
# --- Load the data -------------------------------------------------
# The data loads from the course's public notebook repo, nothing to upload.
# No internet? Set DATA_URL = "" and upload the file when prompted (Colab only).
DATA_URL = "https://raw.githubusercontent.com/eoinlane/BUU33803-student-notebooks/main/tutorial1/data/bike_rentals_tutorial1.csv"

import pandas as pd

if DATA_URL:
    bikes = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload bike_rentals_tutorial1.csv …")
        up = files.upload()
        bikes = pd.read_csv(next(iter(up)))
    except ImportError:
        bikes = pd.read_csv("data/bike_rentals_tutorial1.csv")   # local fallback (repo: data/bike_rentals_tutorial1.csv)

bikes.head()

## What the columns mean

Each row is **one day** (731 in total, two full years).

| Column | Meaning |
|---|---|
| `date` | the calendar date |
| `temp_c` | that day's temperature, in **°C** (the original data records it normalised 0–1; it's been converted here so the numbers mean something) |
| `rentals` | bikes rented that day |

## Step 1, Predict before you compute  🎯

**Before you run anything**, commit to a guess. Write it in the cell below, out loud is even better. No peeking at the data first.

- Will a warmer day mean **more** rentals, less, or no clear pattern? (the *sign*)
- Do you expect a **strong** relationship or a weak one?
- If you fit `rentals = b0 + b1 × temp_c`, what's a plausible value for the **slope `b1`**, and what would it *mean* in plain English?

In [ ]:
# Your prediction (just type it here as notes, this is your commitment):
#
# Sign of the slope         :
# Strong or weak            :
# What the slope would mean :

## Step 2, Explore with Claude

Look at the data *before* modelling. A good prompt to Claude:

> *"Here's 731 days of temperature and bike rentals. Plot rentals against temperature and describe the relationship in plain English, is it roughly linear? Any obvious outliers?"*

Here's the kind of code it will give you, run it:

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(bikes['temp_c'], bikes['rentals'], s=10, alpha=0.5)
plt.xlabel('Temperature (°C)')
plt.ylabel('Bikes rented that day')
plt.title('Bike rentals vs temperature')
plt.show()

bikes.describe()

Does the picture match the prediction you committed to in Step 1? Don't change your guess, we want to see whether the *model* agrees. (One thing you'll probably notice: the points don't hug a tight line the way Armand's Pizza did. Hold that thought for Step 6.)

## Step 3, Build the model  (let Claude drive)

Now fit the line. A good prompt to Claude:

> *"Fit a simple linear regression of `rentals` on `temp_c` in Python using statsmodels, and print the full summary so I can read the coefficient, its p-value, and R²."*

Run the code it gives you (the same recipe as Armand's Pizza):

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols('rentals ~ temp_c', data=bikes).fit()
print(model.summary())

## Step 4, Reveal & reflect

Now **read the output**. In the summary, find:

1. **The slope**, the `coef` on `temp_c`. What sign? Roughly what size?
2. **Significance**, its `P>|t|`. Real effect, or could it be chance?
3. **R²**, the `R-squared` line. How much of the variation does the model explain?

**Were you right?** Compare each to the guess you committed to in Step 1. What surprised you?

<details><summary>✅ Stuck? What you should find</summary>

Slope ≈ **162**, intercept ≈ **1,215**, R² ≈ **0.39**, p-value effectively **0** (nowhere near chance, with 731 days that's easy). In plain English: **each extra 1°C is worth about 162 more rentals that day**, and the relationship is real, but it only explains **39%** of the ups and downs, well under Armand's Pizza's 90%. Something else is going on.

</details>

## Step 5, Turn it into a business decision

A model is only useful if it changes a decision. **Tomorrow's forecast says 15°C.** Use your model to predict rentals, then decide.

> Ask Claude: *"Using this model, how many rentals would you predict for a 15°C day, and how confident should I be in that single prediction?"*

In [ ]:
pred = model.predict({'temp_c': [15]})
print('Predicted rentals:', round(float(pred.iloc[0])))

How many spare bikes would you have ready, exactly that number, or a range either side? What *else* (not in this dataset) would you want to know before staffing up, a weekend, a public holiday, rain forecast?

## Step 6, Interrogate it  (don't trust blindly)

Push back at least once. Worth asking Claude:

- *"R² is only 0.39, so 61% is unexplained. What kind of thing might a single temperature reading be missing here?"*
- *"This model uses two full years of data, all mixed together. Does a single slope make sense for a scheme that presumably has seasons, and grows in popularity over time?"*
- *"What would make this model **wrong**?"*  ← Tuesday's session is built to answer exactly this, on this same model.

Catching the limits of a model is the real skill, not fitting the line, Claude already did that in one line of code.

## Before you leave

Jot these down, your TA will want them:
1. **Your Step-1 prediction vs. what you found**, were you surprised?
2. The **slope in one plain-English sentence** (what it means for the business).
3. **One reason 39% feels low**, your best guess, even if you can't prove it yet.